# Part 2: CNN Image Classification

从 `Conv2d` 到 EuroSAT 遥感场景分类。

这一章把 Part 1 的训练闭环迁移到真实图像任务上：输入从 `[B, 2]` 变成 `[B, C, H, W]`，标签仍然是 `[B]`。

## 1. 本章目标

- 理解 `nn.Conv2d` 的输入输出 shape
- 写一个最小 CNN 图像分类器
- 复用训练、验证、checkpoint 和曲线绘制流程
- 用 EuroSAT 做遥感场景分类
- 画混淆矩阵，分析类别混淆

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, random_split


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 2. Conv2d 的 shape

PyTorch 图像张量使用 `[B, C, H, W]`。

`nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)` 会把 `[B, 3, 64, 64]` 变成 `[B, 16, 64, 64]`。

如果后面接 `nn.MaxPool2d(2)`，空间尺寸会从 `64x64` 变成 `32x32`。

In [ ]:
x = torch.randn(4, 3, 64, 64)
conv = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
pool = nn.MaxPool2d(2)

y = conv(x)
z = pool(y)

print("input:", x.shape)
print("after conv:", y.shape)
print("after pool:", z.shape)

## 3. 最小 CNN 分类器

这个模型由三段卷积特征提取、全局平均池化和一个线性分类头组成。

In [ ]:
class TinyCNNClassifier(nn.Module):
    """一个最小 CNN 图像分类器。"""

    def __init__(self, in_channels=3, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


model = TinyCNNClassifier(in_channels=3, num_classes=10).to(device)
logits = model(torch.randn(4, 3, 64, 64).to(device))
print("logits:", logits.shape)

In [ ]:
def print_model_shapes(model, image_size=(4, 3, 64, 64)):
    y = torch.randn(*image_size)
    print("input", y.shape)
    with torch.no_grad():
        for layer in model.features.cpu():
            y = layer(y)
            print(layer.__class__.__name__, y.shape)
        y = model.pool(y)
        print("AdaptiveAvgPool2d", y.shape)
        y = torch.flatten(y, 1)
        print("flatten", y.shape)
        y = model.classifier(y)
        print("logits", y.shape)


print_model_shapes(TinyCNNClassifier())
model = model.to(device)

## 4. EuroSAT 数据集

EuroSAT 基于 Sentinel-2 影像，包含 10 个土地利用/土地覆盖类别和 27000 张标注图像。原始版本有 13 个光谱波段，入门实验通常先使用 RGB 版本。

类别包括：`AnnualCrop`、`Forest`、`HerbaceousVegetation`、`Highway`、`Industrial`、`Pasture`、`PermanentCrop`、`Residential`、`River`、`SeaLake`。

如果当前环境不能下载数据，下面的代码会自动回退到 `FakeData`，这样可以先验证 CNN 训练流程。

In [ ]:
try:
    from torchvision import datasets, transforms
    from torchvision.datasets import FakeData
except Exception as exc:
    raise RuntimeError("This notebook needs torchvision. Install torchvision first.") from exc


train_transform = transforms.Compose([
    transforms.RandomResizedCrop(64, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3809, 0.4082], std=[0.2037, 0.1366, 0.1148]),
])

eval_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.3444, 0.3809, 0.4082], std=[0.2037, 0.1366, 0.1148]),
])


def load_eurosat_or_fake(root="data/eurosat", use_fake_if_download_fails=True):
    try:
        base_dataset = datasets.EuroSAT(root=root, transform=train_transform, download=True)
        class_names = base_dataset.classes
        print("Loaded EuroSAT:", len(base_dataset), "images")
        return base_dataset, class_names
    except Exception as exc:
        if not use_fake_if_download_fails:
            raise
        print("EuroSAT download/load failed, using FakeData for code validation.")
        print("Reason:", type(exc).__name__, exc)
        class_names = [
            "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
            "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
        ]
        fake_dataset = FakeData(
            size=1000,
            image_size=(3, 64, 64),
            num_classes=len(class_names),
            transform=train_transform,
        )
        return fake_dataset, class_names


dataset, class_names = load_eurosat_or_fake()
print(class_names)

## 5. 训练/验证划分

教学 notebook 用一个较小子集加快演示。正式实验可以去掉 `Subset`。

In [ ]:
max_samples = min(3000, len(dataset))
dataset = Subset(dataset, list(range(max_samples)))

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42),
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print("images:", images.shape)
print("labels:", labels.shape)

## 6. 训练函数

这部分和 Part 1 几乎一样，只是模型输入换成了图像。

In [ ]:
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return {"loss": total_loss / total_count, "acc": total_correct / total_count}


@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for images, labels in dataloader:
        images = images.to(device)
        labels = labels.to(device)

        logits = model(images)
        loss = F.cross_entropy(logits, labels)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_count += batch_size

    return {"loss": total_loss / total_count, "acc": total_correct / total_count}

In [ ]:
def fit(model, train_loader, val_loader, optimizer, device, num_epochs=5):
    history = []
    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, num_epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = evaluate(model, val_loader, device)
        row = {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_acc": train_metrics["acc"],
            "val_loss": val_metrics["loss"],
            "val_acc": val_metrics["acc"],
        }
        history.append(row)

        if row["val_acc"] > best_val_acc:
            best_val_acc = row["val_acc"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"epoch {epoch:02d} | "
            f"train loss {row['train_loss']:.4f} acc {row['train_acc']:.3f} | "
            f"val loss {row['val_loss']:.4f} acc {row['val_acc']:.3f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    return history

## 7. 先 overfit 小数据

如果模型不能 overfit 32 张图，通常说明 shape、label、loss 或学习率有问题。

In [ ]:
small_dataset = Subset(dataset, list(range(min(32, len(dataset)))))
small_loader = DataLoader(small_dataset, batch_size=16, shuffle=True, num_workers=0)

set_seed(123)
small_model = TinyCNNClassifier(num_classes=len(class_names)).to(device)
small_optimizer = torch.optim.Adam(small_model.parameters(), lr=1e-3)
small_history = fit(small_model, small_loader, small_loader, small_optimizer, device, num_epochs=3)

## 8. 正式训练一个小模型

教学环境先跑 5 个 epoch。真实实验可以增加 epoch、使用完整数据集，并记录 checkpoint。

In [ ]:
set_seed(42)
model = TinyCNNClassifier(num_classes=len(class_names)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = fit(model, train_loader, val_loader, optimizer, device, num_epochs=5)

In [ ]:
def plot_history(history, title="TinyCNN training curve"):
    epochs = [row["epoch"] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="val")
    axes[0].set_title("loss")
    axes[0].legend()
    axes[1].plot(epochs, [row["train_acc"] for row in history], label="train")
    axes[1].plot(epochs, [row["val_acc"] for row in history], label="val")
    axes[1].set_title("accuracy")
    axes[1].legend()
    fig.suptitle(title)
    plt.show()


plot_history(history)

## 9. 混淆矩阵

遥感场景分类中，`Industrial` / `Residential`、`AnnualCrop` / `PermanentCrop` 等类别容易混淆。混淆矩阵能帮你看到错误结构。

In [ ]:
def compute_confusion_matrix(model, dataloader, device, num_classes):
    matrix = torch.zeros(num_classes, num_classes, dtype=torch.long)
    model.eval()
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            labels = labels.to(device)
            preds = model(images).argmax(dim=1)
            for target, pred in zip(labels.cpu(), preds.cpu()):
                matrix[target, pred] += 1
    return matrix


cm = compute_confusion_matrix(model, val_loader, device, len(class_names))
print(cm)

In [ ]:
def plot_confusion_matrix(cm, class_names):
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(cm.numpy(), cmap="Blues")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names, rotation=90)
    ax.set_yticklabels(class_names)
    ax.set_xlabel("predicted")
    ax.set_ylabel("target")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()


plot_confusion_matrix(cm, class_names)

## 10. 遥感迁移

- RGB EuroSAT 是入门版本，真实 Sentinel-2 有 13 个光谱波段。
- 如果输入 13 波段，第一层要从 `nn.Conv2d(3, ...)` 改成 `nn.Conv2d(13, ...)`。
- 场景分类不只看单个物体，还看纹理、空间布局和上下文。
- 后续进入分割任务时，标签会从 `[B]` 变成 `[B, H, W]`。

## 11. 作业

1. 打印每一层 CNN 的 shape，并解释为什么变化。
2. 把通道数从 `16, 32, 64` 改成 `8, 16, 32`，比较速度和精度。
3. 去掉 `BatchNorm2d`，观察训练稳定性。
4. 对比 Adam 和 SGD。
5. 画混淆矩阵，写出最容易混淆的 3 对类别。
6. 思考：如果输入 Sentinel-2 13 波段，模型和归一化参数要怎么改？